# Gradient Boosting con la misma muestra inicial

**Pregunta predictiva del proyecto:** Dado un préstamo que Lending Club acaba de aprobar, y usando únicamente la información disponible el día del desembolso, ¿qué probabilidad hay de que termine declarado incobrable (`Charged Off`) en lugar de pagarse por completo (`Fully Paid`)?

## 1. Objetivo y estado

Comparar HistGradientBoosting con la logística y el árbol sobre las mismas observaciones del experimento inicial. Este notebook está preparado, **sin entrenar ni ejecutar la búsqueda nueva**. Las métricas existentes en `outputs/metrics_boosting.json` corresponden a 5,002 préstamos y permanecen intactas; no son resultados de esta ejecución.

In [ ]:
from pathlib import Path
import sys, os
candidatos_raiz = [Path(os.environ.get("PROYECTO_ML_RAIZ", Path.cwd())).resolve()]
if "__file__" in globals():
    candidatos_raiz.append(Path(__file__).resolve().parent)
raiz_encontrada = next((p for base in candidatos_raiz for p in [base, *base.parents]
                        if (p / "src/config.py").is_file()), None)
if raiz_encontrada is None:
    raise FileNotFoundError("Abra Jupyter dentro del proyecto o defina PROYECTO_ML_RAIZ.")
if str(raiz_encontrada) not in sys.path:
    sys.path.insert(0, str(raiz_encontrada))
from src.rutas import raiz_proyecto
RAIZ = raiz_proyecto(raiz_encontrada)
carpeta = RAIZ


## 2. Muestra y particiones

Objetivo: 220,000 préstamos; tamaño efectivo esperado: 220,003. Se usa `src.data.escanear_y_muestrear`, igual que los scripts del baseline y árbol: estados exactos `Fully Paid`/`Charged Off`, bloques de 50,000, fracción común sin reemplazo y `random_state=42+número_de_bloque`. El baseline del notebook selecciona filas con esa misma receta aunque lea menos columnas.

| Conjunto | Años | Filas esperadas |
|---|---|---:|
| Entrenamiento | 2007–2014 | 73,701 |
| Validación | 2015–2016 | 109,382 |
| Prueba | 2017–2018 | 36,920 |

Antes de entrenar, `verificar_muestra` reconstruye la selección mediante una lectura adicional de solo ID, fecha y estado; exige coincidencia de IDs y orden, fechas, tamaños y prevalencias respecto de las referencias. Guarda huellas por partición. Las métricas antiguas no contienen IDs ni hash del CSV, por lo que no permiten demostrar por sí solas la identidad histórica. La comparación nueva garantiza las mismas observaciones porque recalcula logística, árbol y boosting sobre los mismos objetos de partición.

## 3. Preparación y control temporal

Se mantienen los criterios del experimento inicial: exclusión de datos posteriores, sin precio/calificación de LC y descarte de predictoras con más del 95 % de faltantes **en entrenamiento**. Se esperan 65 variables originales. La fecha de historial se transforma con las funciones compartidas. Los avisos de coherencia no eliminan filas.

La logística usa su imputación, codificación y escalado originales. El árbol y Boosting usan NaN numérico nativo y One-Hot categórico con frecuencia mínima 0.5 %, ajustado solo en entrenamiento. Boosting no escala numéricas. Se mantienen diferencias de preparación propias de cada algoritmo.

Las matrices de logística, árbol y Boosting se construyen de forma secuencial y se liberan las anteriores. La búsqueda de Boosting es secuencial y reutiliza sus matrices; no se paralelizan 108 ajustes ni se reduce la muestra. La importancia por permutación usa hasta 30,000 préstamos de validación, como en el experimento original; esto no reduce el entrenamiento ni la validación de métricas.

## 4. Hiperparámetros y configuración

Se conserva la grilla original: learning_rate=[0.03, 0.05, 0.1], max_iter=[100, 200], max_leaf_nodes=[15, 31, 63], min_samples_leaf=[20, 100] y max_depth=[5, 8, None]. Son **108 configuraciones**. Semilla 42 y early_stopping=False. Se ordenan por ROC-AUC de validación, con brecha entrenamiento-validación como desempate.

`MODO="prueba"` es una alternativa explícita de una configuración con las mismas 220,003 observaciones; no equivale a la búsqueda completa y utiliza carpetas separadas. Ninguno de los modos fue ejecutado al preparar este notebook.

In [ ]:
from src.run_boosting import main, rutas_resultados
from src.boosting import GRILLA
from math import prod
TAMANO_OBJETIVO = 220000
MODO = "completo"
SALIDA, FIGURAS = rutas_resultados(TAMANO_OBJETIVO, MODO)
print("Objetivo:", TAMANO_OBJETIVO, "Configuraciones completas:", prod(map(len, GRILLA.values())))
print("Métricas:", SALIDA)
print("Figuras:", FIGURAS)

## 5. Ejecución manual

**Esta celda inicia las lecturas, entrenamientos y búsqueda cuando usted la ejecute.** Recalcula la logística sin indicadores y el árbol con los parámetros ya elegidos en `outputs/metrics_arbol.json`; no repite la búsqueda del árbol. Después selecciona Boosting en validación y evalúa los parámetros fijados en los tres conjuntos. El umbral se elige en validación con la regla original, nunca en test.

El proceso rechaza destinos que ya contengan archivos para proteger resultados anteriores. Para repetirlo, elija carpetas nuevas en `SALIDA` y `FIGURAS`.

In [ ]:
resultados = main(tamano_muestra=TAMANO_OBJETIVO, salida=SALIDA, figuras=FIGURAS, modo=MODO)

## 6. Evaluación de entrenamiento y validación

La tabla contiene únicamente los modelos recalculados sobre la muestra verificada. PR-AUC significa Average Precision. Las filas de prueba se muestran después como evaluación de los candidatos ya fijados, sin escoger parámetros a partir de ellas.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
from src.evaluate import tabla_metricas
registro = json.loads((SALIDA / "metrics_boosting.json").read_text(encoding="utf-8"))
assert registro["filas_muestra"] == 220003 and registro["tamano_muestra_objetivo"] == 220000
tabla = tabla_metricas(registro["resultados"])
display(tabla[tabla["conjunto"].isin(["entrenamiento", "validacion"])])

## 7. Comparación con logística y árbol

In [ ]:
comparacion = tabla[tabla["conjunto"] == "validacion"].copy()
comparacion["diferencia_ROC_AUC_vs_logistica"] = comparacion["ROC-AUC"] - float(comparacion.loc[comparacion["modelo"] == "logistica_sin_indicadores", "ROC-AUC"].iloc[0])
display(comparacion)
print("Parámetros de Boosting:", registro["parametros_elegidos"])
print("Parámetros del árbol fijados previamente:", registro["parametros_arbol_fijados"])

## 8. Visualizaciones

In [ ]:
for nombre in ["14_boosting_busqueda.png", "15_boosting_importancia.png", "16_boosting_comparacion_roc.png"]:
    display(Image(filename=str(FIGURAS / nombre)))
display(pd.read_csv(SALIDA / "importancia_boosting.csv").head(15))

## 9. Evaluación de prueba e interpretación

Comparar la brecha entre entrenamiento y validación, las métricas de discriminación, y el costo de cómputo. Un cambio mínimo no demuestra una mejora significativa. La selección de parámetros y umbral ya está cerrada antes de las filas siguientes; el periodo de prueba fue consultado en experimentos históricos y no constituye una prueba totalmente desconocida durante el desarrollo.

In [ ]:
display(tabla[tabla["conjunto"] == "prueba"])
print("Tiempo total registrado (segundos):", registro["segundos_totales"])

## 10. Conclusiones pendientes

Las conclusiones se redactarán con los resultados de esta ejecución. Todavía no se sabe si Boosting superará a la logística o al árbol sobre las mismas observaciones. Los porcentajes individuales son estimaciones sin la calibración posterior del flujo final.

Para ejecutarlo desde la raíz del proyecto:

```bash
python -m src.run_boosting --muestra 220000
```

Destinos personalizados:

```bash
python -m src.run_boosting --muestra 220000 --salida outputs/gb_220000_repeticion2 --figuras reports/figures/gb_220000_repeticion2
```

Modo de prueba explícito, una combinación sin cambiar observaciones:

```bash
python -m src.run_boosting --muestra 220000 --modo prueba
```